# Figure 4 — CNN for ACP / PCP subtyping

Reproduces the SERSome CNN shown in Figure 4C-D and Figure S7.

**Input** `data/sersome.npz` — built with `python src/build_sersome.py` from the raw
HORIBA exports, not distributed with this repository.

The archived weights `models/cnn_cp_subtyping.pt` reproduce the published numbers
(14/15 = 93.3% on the independent test set). `src/train_cnn.py` retrains the model
from scratch; training is stochastic, so accuracy may vary by one sample.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
from dataset import load_splits, to_cnn_input
from model import SERSomeCNN, load_model

## 1. Load the SERSome tensor and check the cohort split

In [ ]:
s = load_splits("../data/sersome.npz")
for name in ("train", "valid", "test", "ext"):
    if f"{name}_X" in s:
        print(f"{name:<6} {s[name + '_X'].shape}  PCP: {int(s[name + '_y'].sum())}")
print("wavenumber range:", s["shift"].min(), "-", s["shift"].max(), "cm-1")

## 2. Example SERSomes (each sample = 200 single spectra)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ("ACP", "PCP")):
    y = 0 if name == "ACP" else 1
    idx = int(np.where(s["train_y"] == y)[0][0])
    im = ax.imshow(s["train_X"][idx], aspect="auto", origin="lower", cmap="turbo",
                   extent=[s["shift"][0], s["shift"][-1], 1, 200])
    ax.set_title(f"{name} (sample {idx})")
    ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("spectrum index")
    fig.colorbar(im, ax=ax, label="intensity")
plt.tight_layout()

## 3. Load the released weights and evaluate

In [ ]:
model = load_model("../models/cnn_cp_subtyping.pt")
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
with torch.no_grad():
    prob_test = model(torch.from_numpy(to_cnn_input(s["test_X"]))).squeeze(-1).numpy()
    prob_ext  = model(torch.from_numpy(to_cnn_input(s["ext_X"]))).squeeze(-1).numpy()

prob = np.concatenate([prob_test, prob_ext])
truth = np.concatenate([s["test_y"], s["ext_y"]])
pred = (prob > 0.5).astype(int)

print(f"accuracy {np.mean(pred == truth):.4f}  ({np.sum(pred == truth)}/{len(truth)})")
print("internal test  :", np.sum(pred[:9] == truth[:9]), "/ 9")
print("external cohort:", np.sum(pred[9:] == truth[9:]), "/ 6")

## 4. Confusion matrix (Figure 4D)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(truth, pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["ACP", "PCP"])
disp.plot(cmap="Blues", values_format="d")
plt.title("CNN, independent test set (n = 15)")
plt.tight_layout()

## 5. Training curves (`python src/train_cnn.py` writes this file)

In [ ]:
hist = "../training_history.csv"
if os.path.exists(hist):
    import pandas as pd
    df = pd.read_csv(hist)
    df.plot(x="epoch", y=["train_loss", "valid_loss"], figsize=(6, 4))
    plt.ylabel("weighted BCE loss"); plt.tight_layout()
else:
    print("run src/train_cnn.py to generate", hist)